# 08 — Validation & Exploratory Data Analysis
## Vérification de la qualité du dataset complet

**Sections** :
1. Statistiques descriptives du dataset
2. Visualisation des features clés
3. Corrélations features / labels
4. Validation croisée multi-sources
5. Rapport final de qualité

In [ ]:
import os, sys, json
sys.path.insert(0, "/workspace")

import numpy as np
import pandas as pd

from btc_pipeline.storage.gcs_client import StorageClient

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()

## Section 1 — Statistiques descriptives

In [ ]:
# Charger un mois récent pour l'analyse
features_files = sorted(storage.list_files("processed/features_1s/"))
labels_files = sorted(storage.list_files("processed/labels/"))

print(f"Fichiers features : {len(features_files)}")
print(f"Fichiers labels   : {len(labels_files)}")

if features_files:
    # Charger le dernier mois disponible
    last_features = features_files[-1]
    last_labels = labels_files[-1] if labels_files else None

    print(f"\nChargement de {last_features}...")
    df_f = storage.download_parquet(last_features)
    print(f"  Shape: {df_f.shape}")
    print(f"  Colonnes: {len(df_f.columns)}")
    print(f"  Mémoire: {df_f.memory_usage(deep=True).sum()/1e6:.1f} MB")

    print(f"\nStatistiques descriptives (features numériques) :")
    desc = df_f.describe().T
    desc["nan_pct"] = df_f.isna().mean() * 100
    print(desc[["mean", "std", "min", "50%", "max", "nan_pct"]].round(4).to_string())

    if last_labels:
        df_l = storage.download_parquet(last_labels)
        print(f"\nLabels shape: {df_l.shape}")
        print(df_l.describe().round(6).to_string())
else:
    print("⚠️  Aucun dataset processé trouvé — lancer 07_dataset_builder d'abord")

## Section 2 — Visualisations

In [ ]:
# Visualisations (si matplotlib disponible)
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    if 'df_f' in dir() and not df_f.empty:
        fig, axes = plt.subplots(3, 2, figsize=(14, 10))
        fig.suptitle("Feature Distributions", fontsize=14)

        plot_cols = ["close", "volume", "buy_ratio", "volatility_60s", "trade_count", "price_return"]
        for ax, col in zip(axes.flatten(), plot_cols):
            if col in df_f.columns:
                data = df_f[col].dropna()
                if len(data) > 10000:
                    data = data.sample(10000)
                ax.hist(data, bins=50, alpha=0.7)
                ax.set_title(col)
                ax.set_ylabel("count")

        plt.tight_layout()
        plt.savefig("/workspace/feature_distributions.png", dpi=100)
        print("✅ Graphique sauvegardé : /workspace/feature_distributions.png")
        plt.show()
except ImportError:
    print("matplotlib non disponible — pip install matplotlib pour les visualisations")

## Section 3 — Corrélations features / labels

In [ ]:
if 'df_f' in dir() and 'df_l' in dir() and not df_f.empty:
    # Corrélation de chaque feature avec mu_1m
    label_col = "mu_1m"
    if label_col in df_l.columns:
        numeric_cols = df_f.select_dtypes(include=[np.number]).columns.tolist()
        if "timestamp_s" in numeric_cols:
            numeric_cols.remove("timestamp_s")

        merged = pd.concat([df_f[numeric_cols].reset_index(drop=True),
                            df_l[[label_col]].reset_index(drop=True)], axis=1)
        corr = merged.corr()[label_col].drop(label_col).abs().sort_values(ascending=False)

        print(f"Top 20 corrélations avec {label_col} :")
        print(corr.head(20).to_string())

        print(f"\nBottom 10 (features les moins corrélées) :")
        print(corr.tail(10).to_string())
else:
    print("⚠️  Données non chargées")

## Section 4 — Rapport de qualité

In [ ]:
# Charger les rapports de qualité
quality_files = storage.list_files("processed/validation_reports/")
print(f"Rapports de qualité : {len(quality_files)}")

all_passed = True
for f in quality_files[-5:]:
    try:
        report = storage.download_json(f)
        status = "✅" if report.get("passed") else "❌"
        print(f"  {status} {f}: {report.get('n_rows', 0):,} rows, {report.get('n_features', 0)} features")
        if not report.get("passed"):
            all_passed = False
            for check_name, check_data in report.get("checks", {}).items():
                if isinstance(check_data, dict) and not check_data.get("passed", True):
                    print(f"       ↳ FAILED: {check_name} = {check_data.get('value')} (threshold: {check_data.get('threshold')})")
    except Exception as e:
        print(f"  ⚠️  {f}: {e}")

if all_passed:
    print("\n✅ Tous les contrôles qualité sont passés")
else:
    print("\n⚠️  Certains mois ont des problèmes de qualité")